In [1]:
import ssl
import sys
from pathlib import Path

_original_load_default_certs = ssl.SSLContext.load_default_certs

def _safe_load_default_certs(self, purpose=ssl.Purpose.SERVER_AUTH):
    try:
        return _original_load_default_certs(self, purpose)
    except ssl.SSLError:
        try:
            self.set_default_verify_paths()
        except Exception:
            pass

ssl.SSLContext.load_default_certs = _safe_load_default_certs

ROOT = Path.cwd()

if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

ROOT: C:\Users\Soumadipta Konar\Desktop\fitrank-ai


In [2]:
from src.data_loader import get_bundle_paths, read_docx, load_candidates
from src.semantic_ranker import compute_semantic_scores
from src.lexical_ranker import compute_lexical_scores
from src.features import add_feature_scores
from src.scoring import add_final_scores
from src.submission import create_submission, validate_submission

C:\Users\Soumadipta Konar\anaconda3\envs\torchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "output"

OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "submission.csv"
DEBUG_PATH = OUT_DIR / "top100_debug.csv"

print("RAW_DIR:", RAW_DIR)
print("OUT_PATH:", OUT_PATH)
print("DEBUG_PATH:", DEBUG_PATH)

RAW_DIR: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw
OUT_PATH: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\output\submission.csv
DEBUG_PATH: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\output\top100_debug.csv


In [4]:
paths = get_bundle_paths(RAW_DIR)

candidate_path = paths["candidates"]
jd_path = paths["job_description"]
validator_path = paths["validator"]

print("Candidates:", candidate_path)
print("Job Description:", jd_path)
print("Validator:", validator_path)

Candidates: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\candidates.jsonl
Job Description: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\job_description.docx
Validator: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\raw\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\validate_submission.py


In [5]:
jd_text = read_docx(jd_path)

print("JD length:", len(jd_text))


JD length: 9564


In [6]:
df = load_candidates(candidate_path)

print("Shape:", df.shape)
df.head()

Loading candidates: 100000it [00:20, 4873.59it/s]


Shape: (100000, 14)


,candidate_id,title,headline,summary,location,country,industry,company,years_experience,skills,career_history,education,signals,semantic_text
0,CAND_0000001,Backend Engineer,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Toronto,Canada,IT Services,Mindtree,6.9,"[{'name': 'Tailwind', 'proficiency': 'intermed...","[{'company': 'Mindtree', 'title': 'Backend Eng...",[{'institution': 'Lovely Professional Universi...,"{'profile_completeness_score': 86.9, 'signup_d...",current role: backend engineer headline: backe...
1,CAND_0000002,Operations Manager,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,"Chennai, Tamil Nadu",India,IT Services,Wipro,12.5,"[{'name': 'Project Management', 'proficiency':...","[{'company': 'Wipro', 'title': 'Operations Man...","[{'institution': 'Local Engineering College', ...","{'profile_completeness_score': 78.7, 'signup_d...",current role: operations manager headline: ope...
2,CAND_0000003,Customer Support,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Austin,USA,IT Services,TCS,1.1,"[{'name': 'Angular', 'proficiency': 'intermedi...","[{'company': 'TCS', 'title': 'Customer Support...","[{'institution': 'Local Engineering College', ...","{'profile_completeness_score': 31.9, 'signup_d...",current role: customer support headline: custo...
3,CAND_0000004,Marketing Manager,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Sydney,Australia,Paper Products,Dunder Mifflin,3.8,"[{'name': 'Node.js', 'proficiency': 'intermedi...","[{'company': 'Dunder Mifflin', 'title': 'Marke...","[{'institution': 'Local Engineering College', ...","{'profile_completeness_score': 28.5, 'signup_d...",current role: marketing manager headline: mark...
4,CAND_0000005,Accountant,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,"Gurgaon, Haryana",India,Manufacturing,Stark Industries,11.0,"[{'name': 'SQL', 'proficiency': 'beginner', 'e...","[{'company': 'Stark Industries', 'title': 'Acc...","[{'institution': 'Chandigarh University', 'deg...","{'profile_completeness_score': 84.6, 'signup_d...",current role: accountant headline: accountant ...


In [7]:
df["semantic_score"] = compute_semantic_scores(
    df=df,
    jd_text=jd_text,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device="cuda",
    batch_size=256,
    top_k=25000,
)

df[["candidate_id", "title", "years_experience", "location", "semantic_score"]].sort_values(
    "semantic_score",
    ascending=False
).head(20)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding device: cuda


Batches: 100%|██████████| 391/391 [03:23<00:00,  1.92it/s]


,candidate_id,title,years_experience,location,semantic_score
12442,CAND_0012443,Operations Manager,13.5,Singapore,0.667591
87663,CAND_0087664,Java Developer,7.2,Seattle,0.658735
80852,CAND_0080853,Software Engineer,6.6,"Chandigarh, Chandigarh",0.658627
20653,CAND_0020654,Project Manager,6.7,"Coimbatore, Tamil Nadu",0.658056
56624,CAND_0056625,Civil Engineer,11.9,"Trivandrum, Kerala",0.656654
64351,CAND_0064352,Sales Executive,5.5,"Delhi, Delhi",0.654650
69083,CAND_0069084,Software Engineer,7.9,San Francisco,0.654466
45488,CAND_0045489,Java Developer,5.2,Sydney,0.651983
35587,CAND_0035588,Civil Engineer,6.5,"Kochi, Kerala",0.651926
95452,CAND_0095453,Graphic Designer,7.4,"Kochi, Kerala",0.651334


In [8]:
df["lexical_score"] = compute_lexical_scores(df)

df[["candidate_id", "title", "years_experience", "lexical_score"]].sort_values(
    "lexical_score",
    ascending=False
).head(20)

,candidate_id,title,years_experience,lexical_score
55904,CAND_0055905,Senior Machine Learning Engineer,8.1,1.000000
81845,CAND_0081846,Lead AI Engineer,6.7,0.994876
18498,CAND_0018499,Senior Machine Learning Engineer,7.2,0.963619
46063,CAND_0046064,Senior NLP Engineer,8.9,0.932398
5259,CAND_0005260,Senior NLP Engineer,5.2,0.883659
41610,CAND_0041611,Staff Machine Learning Engineer,6.4,0.876894
71973,CAND_0071974,Senior AI Engineer,7.8,0.847916
2024,CAND_0002025,Senior AI Engineer,5.9,0.843137
46524,CAND_0046525,Senior Machine Learning Engineer,6.1,0.807180
33860,CAND_0033861,Senior NLP Engineer,8.0,0.790022


In [9]:
df = add_feature_scores(df)

df[[
    "candidate_id",
    "title",
    "years_experience",
    "semantic_score",
    "lexical_score",
    "ir_nlp_score",
    "skill_depth_score",
    "production_score",
    "behavior_score",
    "honeypot_penalty"
]].head()

,candidate_id,title,years_experience,semantic_score,lexical_score,ir_nlp_score,skill_depth_score,production_score,behavior_score,honeypot_penalty
0,CAND_0000001,Backend Engineer,6.9,0.560395,0.219815,0.250,0.85478,0.625,0.6583,0.68
1,CAND_0000002,Operations Manager,12.5,0.575325,0.050892,0.125,0.00000,0.750,0.2659,0.35
2,CAND_0000003,Customer Support,1.1,0.563197,0.006591,0.000,0.00000,0.125,0.1951,0.35
3,CAND_0000004,Marketing Manager,3.8,0.561836,0.064166,0.125,0.00000,0.750,0.1039,0.35
4,CAND_0000005,Accountant,11.0,0.533598,0.016459,0.000,0.00000,0.375,0.2606,0.35


In [10]:
df = add_final_scores(df)

df[[
    "candidate_id",
    "title",
    "years_experience",
    "location",
    "country",
    "final_score_raw",
    "semantic_score",
    "lexical_score",
    "ir_nlp_score",
    "skill_depth_score",
    "production_score",
    "behavior_score",
    "honeypot_penalty"
]].head(30)

,candidate_id,title,years_experience,location,country,final_score_raw,semantic_score,lexical_score,ir_nlp_score,skill_depth_score,production_score,behavior_score,honeypot_penalty
0,CAND_0018499,Senior Machine Learning Engineer,7.2,"Noida, Uttar Pradesh",India,0.898936,0.494096,0.963619,1.0,0.975805,1.0,0.8334,1.0
1,CAND_0002025,Senior AI Engineer,5.9,"Trivandrum, Kerala",India,0.897736,0.583604,0.843137,1.0,0.987760,1.0,0.8240,1.0
2,CAND_0046525,Senior Machine Learning Engineer,6.1,"Pune, Maharashtra",India,0.890126,0.569366,0.807180,1.0,0.934577,1.0,0.8270,1.0
3,CAND_0055905,Senior Machine Learning Engineer,8.1,London,UK,0.886079,0.579327,1.000000,1.0,0.969051,1.0,0.7023,1.0
4,CAND_0011687,Senior NLP Engineer,7.8,"Indore, Madhya Pradesh",India,0.879346,0.518832,0.760752,1.0,1.001502,1.0,0.8411,1.0
5,CAND_0081846,Lead AI Engineer,6.7,"Jaipur, Rajasthan",India,0.875645,0.567865,0.994876,1.0,0.962231,1.0,0.6401,1.0
6,CAND_0077337,Staff Machine Learning Engineer,7.0,"Kochi, Kerala",India,0.872641,0.581954,0.577777,1.0,0.971601,1.0,0.8718,1.0
7,CAND_0008425,Senior NLP Engineer,7.8,"Kolkata, West Bengal",India,0.864912,0.571889,0.744883,1.0,0.977623,1.0,0.7516,1.0
8,CAND_0053591,AI Engineer,5.3,Toronto,Canada,0.859155,0.599402,0.580892,1.0,0.950358,1.0,0.8673,1.0
9,CAND_0005538,Senior AI Engineer,5.9,"Kolkata, West Bengal",India,0.855675,0.568471,0.595644,1.0,0.973694,1.0,0.8728,1.0


In [11]:
submission = create_submission(
    df=df,
    out_path=OUT_PATH,
    debug_path=DEBUG_PATH,
)

print("Saved submission:", OUT_PATH)
print("Saved debug:", DEBUG_PATH)

submission.head(20)

Saved submission: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\output\submission.csv
Saved debug: C:\Users\Soumadipta Konar\Desktop\fitrank-ai\data\output\top100_debug.csv


,candidate_id,rank,score,reasoning
0,CAND_0018499,1,0.995000,Senior Machine Learning Engineer with 7.2 year...
1,CAND_0002025,2,0.991144,Senior AI Engineer with 5.9 years experience; ...
2,CAND_0046525,3,0.966677,Senior Machine Learning Engineer with 6.1 year...
3,CAND_0055905,4,0.953668,Senior Machine Learning Engineer with 8.1 year...
4,CAND_0011687,5,0.932022,Senior NLP Engineer with 7.8 years experience;...
5,CAND_0081846,6,0.920124,Lead AI Engineer with 6.7 years experience; re...
6,CAND_0077337,7,0.910469,Staff Machine Learning Engineer with 7.0 years...
7,CAND_0008425,8,0.885622,Senior NLP Engineer with 7.8 years experience;...
8,CAND_0053591,9,0.867114,AI Engineer with 5.3 years experience; relevan...
9,CAND_0005538,10,0.855927,Senior AI Engineer with 5.9 years experience; ...


In [12]:
stdout, stderr = validate_submission(validator_path, OUT_PATH)

print("STDOUT:")
print(stdout)

print("STDERR:")
print(stderr)

STDOUT:
Submission is valid.

STDERR:



In [13]:
import pandas as pd

final = pd.read_csv(OUT_PATH)
debug = pd.read_csv(DEBUG_PATH)

print(final.shape)
print(debug.shape)

final.head()

(100, 4)
(100, 16)


,candidate_id,rank,score,reasoning
0,CAND_0018499,1,0.995000,Senior Machine Learning Engineer with 7.2 year...
1,CAND_0002025,2,0.991144,Senior AI Engineer with 5.9 years experience; ...
2,CAND_0046525,3,0.966677,Senior Machine Learning Engineer with 6.1 year...
3,CAND_0055905,4,0.953668,Senior Machine Learning Engineer with 8.1 year...
4,CAND_0011687,5,0.932022,Senior NLP Engineer with 7.8 years experience;...
